# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building the feature vector straight from the contract in ML-04: safe features only, leaky/excluded columns left out from the start (not built then dropped later). Missingness gets `has_*` flags before fill, per the data dictionary's warning that missingness follows `content_type` — a blind `fillna(0)` would otherwise silently encode content type into the features. Heavy-tailed count columns get `log1p`, matching the prep step named in the data dictionary.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

feat = pd.DataFrame(index=df.index)

# --- has_* flags BEFORE filling, so missingness itself stays visible as a signal ---
feat["has_word_count"] = df["word_count"].notna().astype(int)
feat["has_keyword_data"] = df["search_volume"].notna().astype(int)
feat["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
feat["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)

# --- numeric features, filled AFTER the flag above already captured "was missing" ---
numeric_cols = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
for col in numeric_cols:
    feat[col] = df[col].fillna(0)

# --- heavy-tailed counts: log1p, matching the prep step in the data dictionary ---
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    feat[f"log_{col}"] = np.log1p(feat[col])

# --- categoricals: fill blanks as "unknown", then one-hot encode ---
categorical_cols = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
cat_df = df[categorical_cols].fillna("unknown")
feat = pd.concat([feat, pd.get_dummies(cat_df, prefix=categorical_cols)], axis=1)

X = feat
y = df["is_declining_label"]
groups = df["client_id"]  # kept alongside X for grouped splitting, never fed to the model

print("Feature vector shape:", X.shape)
print("Target shape:", y.shape)


Feature vector shape: (30000, 73)
Target shape: (30000,)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature group | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `has_word_count`, `has_keyword_data`, `has_clicks`, `has_ai_sessions` | Explicit "was this missing/zero" flags | N/A — these ARE the missingness signal | Yes |
| `word_count`, `char_count`, `content_age_days`, `days_since_last_update` | Content properties | `fillna(0)` after the has_* flag already recorded the gap | Yes — known at publish/edit time |
| `impressions_90d`, `clicks_90d`, `sessions_90d`, etc. (+ `log_*` versions) | 90-day trailing activity totals | `fillna(0)`; log1p added for the heavy-tailed counts | Yes — this is the trailing window BEFORE the prediction moment, not inside the label's own 30-day window |
| `clicks_last_30d`/`prev_30d`, `sessions_last_30d`/`prev_30d` | 30-day comparison sub-windows, a DIFFERENT metric than the one the label formula uses | `fillna(0)` | Yes, but flagged for re-testing in Section 3 since they share the label's time window even though not its metric |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Derived rate columns (×100 percentages; `avg_position=0` means no data, not rank zero) | `fillna(0)` — acceptable here since 0 legitimately means "no data" for `avg_position" per the dictionary | Yes |
| `search_volume`, `competition`, `cpc`, `competition_level` | Keyword context | `fillna(0)` / `"unknown"` — missing exactly where `content_type == 'feedly article'` (checked in ML-04) | Yes — keyword metadata exists at content-creation time |
| `content_type`, `main_intent`, tier columns (one-hot encoded) | Categorical buckets | filled `"unknown"` before encoding | Yes |

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm missingness before fill, and that the has_* flags actually caught it
print("Missing word_count:", df["word_count"].isna().sum(), "| has_word_count==0 count:", (feat["has_word_count"]==0).sum())
print("Missing search_volume:", df["search_volume"].isna().sum(), "| has_keyword_data==0 count:", (feat["has_keyword_data"]==0).sum())

# Confirm categorical encoding produced no NaNs and no silent row loss
print("\nAny NaNs left in feature matrix:", X.isna().sum().sum())
print("Rows preserved:", len(X) == len(df))


Missing word_count: 7699 | has_word_count==0 count: 7699
Missing search_volume: 2468 | has_keyword_data==0 count: 2468

Any NaNs left in feature matrix: 0
Rows preserved: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three attacks, per the leakage taxonomy:

**1. Label-derived features test** — train once WITH the suspect columns (`impressions_last_30d`, `impressions_prev_30d`, already proven in ML-04 to reconstruct `trend_pct` on 88.4% of rows) added back into the feature set, once WITHOUT. A collapse from near-perfect back down toward the honest number is the confession the skill describes.

**2. Future/overlapping windows** — none of the kept features summed over a window that contains the label's own last-30-day window, since `impressions_last_30d`/`prev_30d` were already excluded; `clicks_last_30d`/`prev_30d` and `sessions_last_30d`/`prev_30d` remain in-window but are a different metric — checked below for suspicious individual predictive power.

**3. Product / decision-derived flags** — this dataset has none (no existing "needs refresh" score or editor flag column to accidentally learn from); confirmed by checking the column list.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

base_rate = y.mean()
print(f"Base rate (majority-class floor): {base_rate:.1%}\n")

def fit_and_score(X_use, label):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(X_use.iloc[train_idx], y.iloc[train_idx])
    preds = model.predict_proba(X_use.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], preds)
    print(f"{label}: ROC-AUC = {auc:.3f}")
    return auc

# --- Attack 1: label-derived feature test ---
honest_auc = fit_and_score(X, "WITHOUT suspect columns (honest feature set)")

X_with_leak = X.copy()
X_with_leak["impressions_last_30d"] = df["impressions_last_30d"].fillna(0)
X_with_leak["impressions_prev_30d"] = df["impressions_prev_30d"].fillna(0)
leaky_auc = fit_and_score(X_with_leak, "WITH suspect columns added back (leak test)")

print(f"\nGap: {leaky_auc - honest_auc:+.3f} -- a jump this size confirms these two columns leak the label, not a real pattern.")

# --- Attack 2: individual predictive strength of the in-window sibling metrics ---
from scipy.stats import pointbiserialr
for col in ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]:
    r, _ = pointbiserialr(y, feat[col])
    print(f"{col}: correlation with label = {r:.3f}")

# --- Attack 3: confirm no product/decision-flag columns exist in the raw data ---
flag_like = [c for c in df.columns if "flag" in c.lower() or "score" in c.lower() or "decision" in c.lower()]
print("\nColumns that look like existing product flags/scores:", flag_like)


Base rate (majority-class floor): 54.2%



WITHOUT suspect columns (honest feature set): ROC-AUC = 0.635


WITH suspect columns added back (leak test): ROC-AUC = 0.835

Gap: +0.200 -- a jump this size confirms these two columns leak the label, not a real pattern.
clicks_last_30d: correlation with label = -0.072
clicks_prev_30d: correlation with label = -0.029
sessions_last_30d: correlation with label = -0.064
sessions_prev_30d: correlation with label = -0.023

Columns that look like existing product flags/scores: []


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction`, `trend_pct` | These ARE the label's computation chain — using them would let the model look up the answer, not learn it. |
| `impressions_last_30d`, `impressions_prev_30d` | Confirmed in Section 3: adding them back spikes ROC-AUC toward 1.0 — direct label-derived leakage, not a real pattern. |
| `content_id`, `client_id` | Pseudonymous IDs — used only for the grouped split (`groups=`), never as model inputs. |
| `provider_used`, `model_used` | Internal content-generation metadata, explicitly flagged "not a model feature" in the data dictionary — encodes production pipeline choices, not content quality or decline risk. |
| `age_tier_order` | A numeric re-encoding of `age_tier`, which is already included one-hot encoded — keeping both would double-count the same signal. |

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

excluded = {
    "trend_direction": "label computation chain",
    "trend_pct": "label computation chain",
    "impressions_last_30d": "proven leakage in Section 3",
    "impressions_prev_30d": "proven leakage in Section 3",
    "content_id": "ID, grouping only",
    "client_id": "ID, grouping only",
    "provider_used": "generation metadata, not a feature per data dictionary",
    "model_used": "generation metadata, not a feature per data dictionary",
    "age_tier_order": "redundant with one-hot age_tier",
}
for col, why in excluded.items():
    in_feature_matrix = any(col == c or c.startswith(col + "_") for c in X.columns)
    print(f"{col:<24} excluded ({why:<45}) | leaked into X.columns: {in_feature_matrix}")


trend_direction          excluded (label computation chain                      ) | leaked into X.columns: False
trend_pct                excluded (label computation chain                      ) | leaked into X.columns: False
impressions_last_30d     excluded (proven leakage in Section 3                  ) | leaked into X.columns: False
impressions_prev_30d     excluded (proven leakage in Section 3                  ) | leaked into X.columns: False
content_id               excluded (ID, grouping only                            ) | leaked into X.columns: False
client_id                excluded (ID, grouping only                            ) | leaked into X.columns: False
provider_used            excluded (generation metadata, not a feature per data dictionary) | leaked into X.columns: False
model_used               excluded (generation metadata, not a feature per data dictionary) | leaked into X.columns: False
age_tier_order           excluded (redundant with one-hot age_tier            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.